In [3]:
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

In [6]:
df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv')

In [7]:
df.head()

,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,5cf39741-d5bd-4226-a94d-ff0c13ca5eaa,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,41196666-5ad7-4651-98aa-9fa9ddb4aad5,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,6b6a74ae-4b78-4a17-a338-5418ac1408cb,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,e94de938-6049-4e4f-9ac1-b6f98884e235,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2943f760-b273-4602-bd66-6e5c73ae0edf,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [8]:
data=df[['question','long_answers']]

In [9]:
context_data=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_test.csv')

In [10]:
context_data.head()

,id,sample_id,title,url,text
0,a17510ca-d855-4390-b8fc-52b237358291,-7013890438520559398,IFFHS,https://en.wikipedia.org/wiki/International%20...,NaN
1,a86226f3-d5f7-4e3b-b025-17897e623442,-7013890438520559398,IFFHS,https://en.wikipedia.org/wiki/International%20...,# # # # table : formation1984 headquarterszuri...
2,007fa2ea-9730-4ced-96fb-7d7ba4dede3b,-7013890438520559398,IFFHS,https://en.wikipedia.org/wiki/International%20...,"##fhs has no affiliation with fifa, [ 8 ] but ..."
3,83cade83-76c4-4622-b675-10fb78b1acac,-7013890438520559398,IFFHS,https://en.wikipedia.org/wiki/International%20...,"# # the world ' s best club since 1991, the en..."
4,638b49a5-ef69-4b61-b1ed-6abae66cb607,-7013890438520559398,IFFHS,https://en.wikipedia.org/wiki/International%20...,# # # # table : club ; wins ; years barcelona ...


In [11]:
context=context_data['text'].dropna().tolist()
context

['# # # # table : formation1984 headquarterszurich, switzerland official language english, french, spanish, german presidentsaleh irfan bahwini [ 1 ] websiteiffhs. com the international federation of football history & statistics ( iffhs ) is an organisation that chronicles the history and records of association football. [ 2 ] [ 3 ] [ 4 ] it was founded in 1984 by alfredo poge in leipzig. [ 2 ] the iffhs was based in abu dhabi for some time but, in 2010, relocated to bonn, germany, and then in 2014 to zurich. [ 5 ] from its early stages to 2002, the iffhs concentrated on publishing the quarterly magazines fußball - weltzeitschrift, libero spezial deutsch and libero international. [ 6 ] when these had to be discontinued for reasons which were not officially told, the organisation published its material in a series of multi - lingual books in co - operation with sponsors. [ 7 ] the statistical organisation has now confined its publishing activities to its website. iffhs has no affiliati

In [12]:
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'

# 모델 로드
model = SentenceTransformer(model_path)

# 모델을 GPU로 이동 (가능한 경우)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [14]:
questions=data['question']

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [18]:
def search_documents(query, documents, model, device="cuda"):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)
    document_embeddings = model.encode(documents, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, document_embeddings)

    top_results = similarities.argsort(descending=True)[:5]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=documents[idx]
        res[tmp]=similarities[idx]
        
    return list(res.items())


for i in range(20):
    print(f"Query {i+1} : {questions[i]}")
    print("-"*100)
    results = search_documents(questions[i], context, model)
    
    print("Retrieved doc :")
    for j in range(len(results)):
        print(f"\tRank {j} : {results[j]}")
        
    prompt = f"""
    Context information is below.
    ---------------------
    {results}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {questions[i]}
    Answer:
    """
    inputs = tokenizer_gen(prompt, return_tensors="pt", truncation=True).to(device)

    inputs['attention_mask'] = (inputs['input_ids'] != tokenizer_gen.pad_token_id).long().to(device)


    with torch.no_grad():
        generated = model_gen.generate(
            inputs.input_ids, 
            attention_mask=inputs.attention_mask,
            pad_token_id=tokenizer_gen.pad_token_id,
            max_new_tokens=256,
        )
    generated_text = tokenizer_gen.decode(generated[:, inputs.input_ids.shape[1]:][0], skip_special_tokens=True)
    print(generated_text)

Query 1 : Who has the highest goals in world football?
----------------------------------------------------------------------------------------------------


tensor([  71,  127,  118,   81, 3797], device='cuda:0')
Retrieved doc :
	Rank 0 : ('# # # # fewest goals scored * china – 0 [ 31 ] * dutch east indies – 0 [ 31 ] * trinidad and tobago – 0 [ 31 ] * zaire – 0 [ 31 ] # # # # highest goal difference * brazil – + 129 [ 31 ] # # # in one tournament # # # # most goals scored * hungary – 27 ( 1954 ) [ s ] [ 157 ] # # # # fewest goals conceded * switzerland – 0 ( 2006 ) [ s ] [ 158 ] # # # # most goals conceded * south korea – 16 ( 1954 ) [ s ] [ 159 ] # # # # most matches gone into extra time * belgium – 3 ( 1986 ) [ 160 ] * england – 3 ( 1990 ) [ 160 ] * argentina – 3 ( 2014 ) [ 160 ] * croatia – 3 ( 2018 ) [ 160 ] # # # # most minutes without conceding a goal * italy – 517 mins ( 1990 ) [ s ] [ 161 ] # # # # highest goal difference * hungary – + 17 ( 1954 ) [ s ] [ 157 ]', tensor(0.6683, device='cuda:0'))
	Rank 1 : ('# # # rsssf statistics as the rsssf uses different methodology from that of the iffhs and other media outlets to determine whi

KeyboardInterrupt: 